# 02 — Création des features et de la target

On part du dataset nettoyé (`data/processed/btc_clean.csv`) et on crée les colonnes utiles au modèle :

- des **features** = ce que le marché donne à l'instant `t` (présent + passé)
- une **target** = `volatility_24h_future`, ce qu'on veut prédire (futur)

> NB: une **feature** ne regarde que le passé/présent. Seule la **target** a le droit de regarder le futur.

## Partie 1 — Imports & chargement des données

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Chemin vers le dataset nettoye produit par Celia (Membre 1)
CLEAN_PATH = Path("../data/processed/btc_clean.csv")

# parse_dates : un CSV ne garde pas les types, on re-convertit les dates en datetime
df = pd.read_csv(CLEAN_PATH, parse_dates=["open_time", "close_time"])

print("Shape :", df.shape)
df.head()

Shape : (4368, 11)


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume
0,2025-12-01 00:00:00,90360.01,90417.00,86959.99,87000.00,4607.45526,2025-12-01 00:59:59.999999,4.075802e+08,742069,1881.95608,1.663911e+08
1,2025-12-01 01:00:00,87000.00,87749.99,86941.40,87168.91,1588.04096,2025-12-01 01:59:59.999999,1.387834e+08,368794,865.41630,7.563708e+07
2,2025-12-01 02:00:00,87168.90,87500.00,86474.34,86722.29,2052.19788,2025-12-01 02:59:59.999999,1.784774e+08,306696,840.04871,7.306030e+07
3,2025-12-01 03:00:00,86722.30,86800.00,86161.61,86346.13,2001.96556,2025-12-01 03:59:59.999999,1.730082e+08,305414,603.75053,5.218520e+07
4,2025-12-01 04:00:00,86346.13,86350.01,85695.44,85801.03,1863.01351,2025-12-01 04:59:59.999999,1.602050e+08,290041,731.14896,6.286958e+07


In [3]:
# Verifications rapides des prerequis (cf. mes dependances envoyees a Celia)
print("Colonnes :", df.columns.tolist())
print("\nTypes :")
print(df.dtypes)
print("\nTrie par date croissante :", df["open_time"].is_monotonic_increasing)
print("Doublons open_time :", df.duplicated(subset=["open_time"]).sum())

Colonnes : ['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_asset_volume', 'number_of_trades', 'taker_buy_base_volume', 'taker_buy_quote_volume']

Types :
open_time                 datetime64[us]
open                             float64
high                             float64
low                              float64
close                            float64
volume                           float64
close_time                datetime64[us]
quote_asset_volume               float64
number_of_trades                   int64
taker_buy_base_volume            float64
taker_buy_quote_volume           float64
dtype: object

Trie par date croissante : True
Doublons open_time : 0


## Partie 2 — `return_1h` (le rendement horaire)

C'est la **brique de base** de tout le projet : la variation de prix d'une heure à l'autre.

**Formule :** `return_1h = (close_t - close_t-1) / close_t-1`

Autrement dit : « de combien (en %) le prix de clôture a bougé par rapport à l'heure précédente ».
- `+0.01` = le prix a monté de 1 %
- `-0.02` = le prix a baissé de 2 %

En pandas, `.pct_change()` calcule exactement ça automatiquement. La **1ʳᵉ ligne sera `NaN`** (pas d'heure précédente pour la comparer) — c'est normal, on gérera tous les `NaN` à la fin.

In [4]:
# return_1h = variation du close d'une heure a l'autre, en pourcentage
df["return_1h"] = df["close"].pct_change()

# Apercu : on regarde close et return_1h cote a cote
df[["open_time", "close", "return_1h"]].head()

,open_time,close,return_1h
0,2025-12-01 00:00:00,87000.00,NaN
1,2025-12-01 01:00:00,87168.91,0.001941
2,2025-12-01 02:00:00,86722.29,-0.005124
3,2025-12-01 03:00:00,86346.13,-0.004338
4,2025-12-01 04:00:00,85801.03,-0.006313


## Partie 3 — Volatilité passée & future (la TARGET)

La **volatilité** = à quel point le prix bouge. On la mesure par l'**écart-type des `return_1h`** sur une fenêtre de 24h.

- `volatility_24h_past` = écart-type des returns sur les **24h passées** (fenêtre `[t-23 ... t]`) → **feature** (info connue à l'instant t)
- `volatility_24h_future` = écart-type des returns sur les **24h suivantes** (fenêtre `[t+1 ... t+24]`) → **TARGET** (ce qu'on veut prédire)

> Astuce pandas : `rolling(24).std()` calcule l'écart-type glissant sur 24 lignes (en regardant **en arrière**). Pour la version *future*, on décale ce résultat de 24 lignes vers le haut avec `.shift(-24)`, ce qui aligne la fenêtre sur le futur. Les deux fenêtres **ne se chevauchent pas** (l'une finit à `t`, l'autre commence à `t+1`) → pas de triche.

In [5]:
# Volatilite passee : ecart-type des returns sur les 24 dernieres heures [t-23 ... t]
df["volatility_24h_past"] = df["return_1h"].rolling(window=24).std()

# Volatilite future (TARGET) : ecart-type des returns sur les 24 heures suivantes [t+1 ... t+24]
# rolling(24).std() regarde en arriere ; .shift(-24) decale le resultat sur le futur
df["volatility_24h_future"] = df["return_1h"].rolling(window=24).std().shift(-24)

df[["open_time", "return_1h", "volatility_24h_past", "volatility_24h_future"]].head(3)

,open_time,return_1h,volatility_24h_past,volatility_24h_future
0,2025-12-01 00:00:00,NaN,NaN,0.005677
1,2025-12-01 01:00:00,0.001941,NaN,0.005659
2,2025-12-01 02:00:00,-0.005124,NaN,0.005572


## Partie 4 — Les autres features (amplitude, volume, activité, temps)

On crée tout le reste d'un coup, regroupé par thème :

| Feature | Formule | Ce que ça mesure |
|---|---|---|
| `high_low_range` | `(high - low) / close` | amplitude du prix dans l'heure |
| `open_close_return` | `(close - open) / open` | sens du mouvement dans l'heure |
| `volume_change` | variation du `volume` | accélération/ralentissement de l'activité |
| `quote_volume_change` | variation du `quote_asset_volume` | idem, en valeur (USDT) |
| `trade_intensity` | `number_of_trades / volume` | nb de trades par unité de volume |
| `buy_pressure` | `taker_buy_base_volume / volume` | part des achats « agressifs » |
| `hour` | heure de `open_time` (0–23) | effet heure de la journée |
| `day_of_week` | jour de `open_time` (0=lundi) | effet jour de la semaine |

Toutes ne regardent que le présent/passé → ce sont bien des **features** valides.

In [6]:
# Amplitude et sens du mouvement dans l'heure (uniquement la bougie t)
df["high_low_range"] = (df["high"] - df["low"]) / df["close"]
df["open_close_return"] = (df["close"] - df["open"]) / df["open"]

# Variations d'activite d'une heure a l'autre (passe -> present)
df["volume_change"] = df["volume"].pct_change()
df["quote_volume_change"] = df["quote_asset_volume"].pct_change()

# Intensite des trades et pression acheteuse (bougie t)
df["trade_intensity"] = df["number_of_trades"] / df["volume"]
df["buy_pressure"] = df["taker_buy_base_volume"] / df["volume"]

# Features temporelles extraites de open_time
df["hour"] = df["open_time"].dt.hour
df["day_of_week"] = df["open_time"].dt.dayofweek

df.head(3)

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,...,volatility_24h_past,volatility_24h_future,high_low_range,open_close_return,volume_change,quote_volume_change,trade_intensity,buy_pressure,hour,day_of_week
0,2025-12-01 00:00:00,90360.01,90417.00,86959.99,87000.00,4607.45526,2025-12-01 00:59:59.999999,4.075802e+08,742069,1881.95608,...,NaN,0.005677,0.039736,-0.037185,NaN,NaN,161.058319,0.408459,0,0
1,2025-12-01 01:00:00,87000.00,87749.99,86941.40,87168.91,1588.04096,2025-12-01 01:59:59.999999,1.387834e+08,368794,865.41630,...,NaN,0.005659,0.009276,0.001941,-0.655332,-0.659494,232.232045,0.544958,1,0
2,2025-12-01 02:00:00,87168.90,87500.00,86474.34,86722.29,2052.19788,2025-12-01 02:59:59.999999,1.784774e+08,306696,840.04871,...,NaN,0.005572,0.011827,-0.005124,0.292283,0.286015,149.447577,0.409341,2,0


## Partie 5 — Gestion des NaN & sauvegarde

Certaines features créent des `NaN` :
- au **début** : `return_1h` (1 ligne), `volatility_24h_past` (23 lignes), `volume_change`… → pas assez d'historique
- à la **fin** : `volatility_24h_future` (24 dernières lignes) → pas assez de futur

On **supprime ces lignes incomplètes** (`dropna`) pour ne garder que les lignes où toutes les colonnes existent, puis on sauvegarde le dataset enrichi dans `data/processed/btc_features.csv`.

In [7]:
# Nombre de NaN par colonne avant nettoyage
print("NaN par colonne :")
print(df.isna().sum())

# On supprime les lignes incompletes (debut + fin)
lignes_avant = len(df)
df_features = df.dropna().reset_index(drop=True)
print(f"\nLignes : {lignes_avant} -> {len(df_features)} ({lignes_avant - len(df_features)} supprimees)")

# Sauvegarde du dataset enrichi
OUT_PATH = Path("../data/processed/btc_features.csv")
df_features.to_csv(OUT_PATH, index=False)
print(f"\nDataset enrichi sauvegarde -> {OUT_PATH}")
print("Shape finale :", df_features.shape)
df_features.head(3)

NaN par colonne :
open_time                  0
open                       0
high                       0
low                        0
close                      0
volume                     0
close_time                 0
quote_asset_volume         0
number_of_trades           0
taker_buy_base_volume      0
taker_buy_quote_volume     0
return_1h                  1
volatility_24h_past       24
volatility_24h_future     24
high_low_range             0
open_close_return          0
volume_change              1
quote_volume_change        1
trade_intensity            0
buy_pressure               0
hour                       0
day_of_week                0
dtype: int64

Lignes : 4368 -> 4320 (48 supprimees)

Dataset enrichi sauvegarde -> ..\data\processed\btc_features.csv
Shape finale : (4320, 22)


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,...,volatility_24h_past,volatility_24h_future,high_low_range,open_close_return,volume_change,quote_volume_change,trade_intensity,buy_pressure,hour,day_of_week
0,2025-12-02 00:00:00,86286.01,86674.53,86184.39,86513.33,887.97566,2025-12-02 00:59:59.999999,7.673838e+07,242555,371.43716,...,0.005677,0.006375,0.005665,0.002634,0.469897,0.467350,273.155010,0.418297,0,1
1,2025-12-02 01:00:00,86513.33,87350.00,86394.00,86454.93,1192.01967,2025-12-02 01:59:59.999999,1.035374e+08,319806,591.24181,...,0.005659,0.006390,0.011058,-0.000675,0.342401,0.349225,268.289197,0.496000,1,1
2,2025-12-02 02:00:00,86454.93,86857.79,86272.55,86554.47,522.02975,2025-12-02 02:59:59.999999,4.523200e+07,237361,301.79457,...,0.005572,0.006418,0.006762,0.001151,-0.562063,-0.563134,454.688646,0.578118,2,1
